In [ ]:
!pip install google-genai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!rm -rf "/content/processed_parts.zip"
!rm -rf "/content/processed_parts"
!cp -r "/content/drive/MyDrive/NZ/University of Waikato/Māori Data Findability Project/downloaded/processed.zip" "/content/processed_parts.zip"
!unzip -q processed_parts.zip -d processed_parts

^C
[processed_parts.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of processed_parts.zip or
        processed_parts.zip.zip, and cannot find processed_parts.zip.ZIP, period.


In [ ]:
!find "/content/processed_parts/processed" -type f -name "*.json" | wc -l

10920


In [ ]:
!find "/content/drive/MyDrive/NZ/University of Waikato/Māori Data Findability Project/downloaded/processed_parts" -type f -name "*.json" | wc -l

2428


In [ ]:
# import requests
# from google.colab import userdata

# api_key = userdata.get('GEMINI_API_KEY')
# url = f"https://aiplatform.googleapis.com/v1/publishers/google/models/gemini-2.5-flash-lite:streamGenerateContent?key={api_key}"
# headers = {"Content-Type": "application/json"}
# payload = {"contents": [{"role": "user", "parts": [{"text": "Explain how AI works in a few words"}]}]}

# response = requests.post(url, headers=headers, json=payload)
# print(response.json())

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# from google import genai
# from google.genai.types import HttpOptions

# client = genai.Client(vertexai=True, project="gen-lang-client-0900489206", location="us-central1")
# response = client.models.generate_content(
#     model="gemini-2.5-flash",
#     contents="How does AI work?",
# )
# print(response.text)

In [ ]:
# from google.genai import Client
# from google.genai import types
# from google.colab import userdata
# import json

# # Only run this block for Vertex AI API
# with Client(vertexai=True, project="gen-lang-client-0900489206", location="us-central1") as client:
#     response = client.models.generate_content(
#         model='gemini-2.5-flash', contents='Why is the sky blue?'
#     )
#     print(response.text)


In [ ]:
# # V1 prompt
# def build_classifier_prompt(doc: str) -> str:
#   return f'''
# You are an expert document classifier specializing in New Zealand Parliament Hansard records.

# Classify the following document into exactly one category.

# Categories:
# 1. maori_origin
# 2. non_maori_origin
# 3. irrelevant

# Rules:
# - Choose exactly one label
# - Base decision only on the text
# - If Māori relevance is ambiguous, prefer non_maori_origin
# - A document is only 'maori_origin' if it specifically discusses Māori people, culture, language, issues, or places. Mere mention of a Māori name (person or place) or term does not qualify it as 'maori_origin'.
# - If text is too short, nonsensical, incomplete, or meaningless, choose irrelevant

# Return only this JSON format:

# {{
#   "label": "maori_origin | non_maori_origin | irrelevant",
#   "confidence": 0.00,
#   "reason": "One short sentence explaining why this label was chosen."
# }}

# Document section:
# """
# {doc}
# """
# '''

In [ ]:
def build_classifier_prompt(doc: str) -> str:
    return f'''You are a specialist in classifying New Zealand Parliament Hansard records.

TASK: Classify the document section below into EXACTLY ONE category.

CATEGORIES:
- maori_origin: Substantive discussion of Māori-related topics
- non_maori_origin: Parliamentary content not primarily about Māori topics
- irrelevant: Non-substantive or unclassifiable content

CLASSIFICATION RULES:

1. maori_origin - Use ONLY when the document substantively discusses:
   • Māori people, communities, or populations
   • Māori culture, language (te reo Māori), traditions, or identity
   • Māori political, legal, social, or historical matters
   • Treaty of Waitangi, Māori land rights, iwi/hapū governance
   • Māori-specific policies, programs, or legislation

   CRITICAL: Mere mentions do NOT qualify:
   • Māori names of people or places
   • Incidental references to Māori individuals
   • Geographic locations with Māori names
   • Brief acknowledgments or greetings

   The content must be ABOUT Māori topics, not just mention them.

2. non_maori_origin - Use for:
   • Parliamentary debates, speeches, questions, or discussions on other topics
   • Legislation or policy not primarily focused on Māori matters
   • General government business where Māori relevance is absent or incidental

3. irrelevant - Use for:
   • Parliamentary procedures (prayers, roll calls, minutes)
   • Housekeeping items (adjournments, point of order)
   • Ceremonial statements with no substantive content
   • Fragments too short or unclear to classify
   • Non-sensical or corrupted text

IMPORTANT GUIDELINES:
- If Māori relevance is ambiguous or peripheral → classify as non_maori_origin
- If you are uncertain between maori_origin and non_maori_origin → choose non_maori_origin
- Base decisions solely on the content provided
- Confidence should reflect how clear-cut the classification is (0.0-1.0)

OUTPUT FORMAT (return ONLY this JSON, no other text):
{{
  "label": "maori_origin | non_maori_origin | irrelevant",
  "confidence": 0.85,
  "reason": "Brief explanation of why this classification was chosen."
}}

DOCUMENT SECTION:
"""
{doc}
"""
'''

In [ ]:
import os
import json
from google import genai
from google.genai import types
import json


client = genai.Client(vertexai=True, project="gen-lang-client-0900489206", location="us-central1")

def classify_hansard_section(text, model_name="gemini-2.5-flash", temperature=0.0, mime_type="application/json"):
    prompt = build_classifier_prompt(text)

    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            response_mime_type=mime_type,
        ),
    )

    return json.loads(response.text)

In [ ]:
import time

def classify_hansard_section_with_retry(
    text,
    model_name="gemini-2.5-flash",
    temperature=0.0,
    mime_type="application/json",
    max_retries=2
):
    backoff_times = [30, 60]

    for attempt in range(max_retries + 1):
        try:
            return classify_hansard_section(text=text, model_name=model_name, temperature=temperature, mime_type=mime_type)
        except Exception as e:
            if attempt < max_retries:
                wait = backoff_times[attempt] if len(backoff_times) < attempt else backoff_times[-1]
                print(f"⚠️ API error: {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print("❌ Failed after retries. Skipping.")
                return None

In [ ]:
import glob
import json
import os
import time
from tqdm import tqdm

def classify_hansard_docs(
    processed_dir,
    classify_func,  # classify_hansard_section_with_retry
    model_name="gemini-2.5-flash",
    temperature=0.0,
    mime_type="application/json",
    requests_per_min=600,
    skip_classified=True,
):
    """
    Classify JSON Hansard documents in a folder.

    - classify_func must already handle retries and backoff internally
    - Rate limited via requests_per_min
    - Progress bar shows ETA + current filename
    """

    files = sorted(glob.glob(os.path.join(processed_dir, "*.json")), reverse=True)
    total_files = len(files)

    sleep_per_request = 60 / requests_per_min
    last_request_time = 0

    with tqdm(files, desc="Classifying documents", unit="file") as pbar:
        for path in pbar:
            filename = os.path.basename(path)
            pbar.set_postfix(file=filename)

            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Skip already-classified docs
            if skip_classified and all(
                k in data for k in (
                    "classification_label",
                    "classification_confidence",
                    "classification_reason",
                )
            ):
                continue

            text = data.get("HpsProceedingContent", "").strip()

            if not text:
                data["classification_label"] = "irrelevant"
                data["classification_confidence"] = 1.0
                data["classification_reason"] = (
                    "Document section is empty or contains no meaningful text."
                )
            else:
                # Rate limit enforcement
                elapsed = time.time() - last_request_time
                if elapsed < sleep_per_request:
                    time.sleep(sleep_per_request - elapsed)

                result = classify_func(text=text, model_name=model_name, temperature=temperature, mime_type=mime_type)
                last_request_time = time.time()

                if result is None:
                    data["classification_label"] = "irrelevant"
                    data["classification_confidence"] = 0.0
                    data["classification_reason"] = (
                        "Classification failed due to repeated API errors."
                    )
                else:
                    data["classification_label"] = result["label"]
                    data["classification_confidence"] = result["confidence"]
                    data["classification_reason"] = result["reason"]

            # Write back to file
            with open(path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"✅ Classification complete for {total_files} documents.")

In [ ]:
PROCESSED_PARTS_PATH = "/content/drive/MyDrive/NZ/University of Waikato/Māori Data Findability Project/downloaded/processed_parts"

classify_hansard_docs(
    processed_dir=PROCESSED_PARTS_PATH,
    classify_func=classify_hansard_section_with_retry,
    model_name="gemini-2.5-flash",
    requests_per_min=600,
    skip_classified=True,
)


Classifying documents:   2%|▏         | 24/1501 [02:44<2:52:32,  7.01s/file, file=hansard_2014-01-30_body_section_0.json]

⚠️ API error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Retrying in 60s...


Classifying documents:   2%|▏         | 31/1501 [04:34<3:53:57,  9.55s/file, file=hansard_2014-01-29_body_section_2.json]

⚠️ API error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Retrying in 60s...


Classifying documents:   3%|▎         | 38/1501 [06:29<4:26:16, 10.92s/file, file=hansard_2014-01-28_body_section_5.json]

...

⚠️ API error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Retrying in 60s...


Classifying documents:  99%|█████████▉| 1487/1501 [3:41:16<01:38,  7.06s/file, file=hansard_2012-03-01_body_section_3.json]

⚠️ API error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Retrying in 60s...


Classifying documents: 100%|██████████| 1501/1501 [3:43:51<00:00,  8.95s/file, file=hansard_2012-02-29_body_section_1.json]

✅ Classification complete for 1501 documents.


In [ ]:
import glob
import os
from tqdm import tqdm

REQUIRED_KEYS = (
    '"classification_label"',
    '"classification_confidence"',
    '"classification_reason"',
)

def move_skipped_hansard_docs_fast(
    input_dir,
    output_dir,
    dry_run=False,
    reverse=False,
):
    files = sorted(
        glob.glob(os.path.join(input_dir, "*.json")),
        reverse=reverse
    )

    total_files = len(files)

    if not dry_run:
        os.makedirs(output_dir, exist_ok=True)

    matched = 0

    with tqdm(files, desc="Scanning documents", unit="file") as pbar:
        for i, path in enumerate(files, 1):
            try:
                with open(path, "r", encoding="utf-8") as f:
                    text = f.read()
            except Exception:
                continue

            # FAST PATH: simple string check
            if all(k in text for k in REQUIRED_KEYS):
                matched += 1

                if not dry_run:
                    dest_path = os.path.join(output_dir, os.path.basename(path))
                    os.replace(path, dest_path)  # faster than shutil.move

            # Reduce tqdm overhead
            if i % 100 == 0:
                pbar.set_postfix(matched=matched)
            pbar.update(1)

    percent = (matched / total_files * 100) if total_files else 0.0

    if dry_run:
        print(
            f"🧪 Dry run complete\n"
            f"  Total files: {total_files}\n"
            f"  Files matching skip criteria: {matched}\n"
            f"  Percentage: {percent:.2f}%"
        )
    else:
        print(
            f"✅ Move complete\n"
            f"  Total files processed: {total_files}\n"
            f"  Files moved: {matched}\n"
            f"  Percentage: {percent:.2f}%\n"
            f"  Destination: {output_dir}"
        )


In [ ]:
move_skipped_hansard_docs_fast(
    input_dir="/content/drive/MyDrive/NZ/University of Waikato/Māori Data Findability Project/downloaded/processed_parts",
    output_dir="/content/drive/MyDrive/NZ/University of Waikato/Māori Data Findability Project/downloaded/moved",
    dry_run=False,
    reverse=True,
)

Scanning documents: 100%|██████████| 1501/1501 [00:17<00:00, 85.22file/s, matched=1500] 

✅ Move complete
  Total files processed: 1501
  Files moved: 1501
  Percentage: 100.00%
  Destination: /content/drive/MyDrive/NZ/University of Waikato/Māori Data Findability Project/downloaded/moved
